In [22]:
import pandas as pd

data1 = pd.read_csv("image_code_output.csv")
data1.head()

,code,image_path
0,271002935.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...
1,10028269.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...
2,10028268.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...
3,10025542.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...
4,10027276.0,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...


In [23]:
data2 = pd.read_csv("AI ML Task Sheet.csv")
data2 = data2.loc[:, ~data2.columns.str.startswith('Unnamed')]
data2

,date,code,qty,rate
0,2026-04-22 14:50:52,500001.0,16.0,1296.0
1,2026-04-22 14:50:52,500001.0,4.0,1295.0
2,2026-04-22 14:50:52,500001.0,16.0,1295.0
3,2026-04-22 14:50:52,500001.0,4.0,1295.0
4,2026-04-22 14:50:52,10029028.0,4.0,1250.0
...,...,...,...,...
994,NaN,NaN,NaN,NaN
995,NaN,NaN,NaN,NaN
996,NaN,NaN,NaN,NaN
997,NaN,NaN,NaN,NaN


In [24]:
merged_data = pd.merge(data1, data2, on='code', how='inner')
merged_data_cleaned = merged_data.dropna()

In [25]:
merged_data_cleaned['code'] = merged_data_cleaned['code'].astype('Int64')
merged_data_cleaned['qty'] = merged_data_cleaned['qty'].astype('Int64')
merged_data_cleaned['date'] = pd.to_datetime(merged_data_cleaned['date'], errors='coerce')

In [26]:
df = merged_data_cleaned.copy()
df = df.groupby('code').agg({
    'image_path': 'first',
    'qty': 'sum',
    'rate': 'first',
    'date': 'last'
}).reset_index()

In [27]:
df

,code,image_path,qty,rate,date
0,500001,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,1032,1296.0,2026-03-10 19:51:05
1,10016728,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,24,875.0,2026-04-27 20:14:28
2,10019275,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,56,1650.0,2026-03-14 12:56:24
3,10021130,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,32,875.0,2025-12-10 16:59:21
4,10021131,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,28,875.0,2026-04-16 12:45:32
...,...,...,...,...,...
121,10029443,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,15,995.0,2026-04-28 15:43:29
122,10029444,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,15,995.0,2026-04-27 19:03:28
123,10029447,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,4,850.0,2026-04-28 19:13:19
124,10029448,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,4,850.0,2026-04-28 19:13:19


In [28]:
import numpy as np  
df['date']        = pd.to_datetime(df['date'])
df['month_sin']   = np.sin(2 * np.pi * df['date'].dt.month / 12)
df['month_cos']   = np.cos(2 * np.pi * df['date'].dt.month / 12)
df['quarter_norm']= df['date'].dt.quarter / 4.0


In [29]:
df

,code,image_path,qty,rate,date,month_sin,month_cos,quarter_norm
0,500001,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,1032,1296.0,2026-03-10 19:51:05,1.000000e+00,6.123234e-17,0.25
1,10016728,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,24,875.0,2026-04-27 20:14:28,8.660254e-01,-5.000000e-01,0.50
2,10019275,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,56,1650.0,2026-03-14 12:56:24,1.000000e+00,6.123234e-17,0.25
3,10021130,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,32,875.0,2025-12-10 16:59:21,-2.449294e-16,1.000000e+00,1.00
4,10021131,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,28,875.0,2026-04-16 12:45:32,8.660254e-01,-5.000000e-01,0.50
...,...,...,...,...,...,...,...,...
121,10029443,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,15,995.0,2026-04-28 15:43:29,8.660254e-01,-5.000000e-01,0.50
122,10029444,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,15,995.0,2026-04-27 19:03:28,8.660254e-01,-5.000000e-01,0.50
123,10029447,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,4,850.0,2026-04-28 19:13:19,8.660254e-01,-5.000000e-01,0.50
124,10029448,unzipped_data/1/WhatsApp Image 2026-04-29 at 1...,4,850.0,2026-04-28 19:13:19,8.660254e-01,-5.000000e-01,0.50


In [30]:
import cv2
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = 224

base_model = tf.keras.applications.MobileNetV2(
    include_top=False, weights='imagenet', input_shape=(224, 224, 3)
)
base_model.trainable = False

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.10),
])



In [ ]:
N_AUG = 5

df = df.reset_index(drop=True)
rate_max = df['rate'].max()

features_list = []
meta_list = []
targets_list = []

for i in range(len(df)):
    path = df.loc[i, 'image_path']
    img  = cv2.imread(path)
    if img is None:
        print(f"Skipping {path}")
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    meta = [
        df.loc[i, 'rate'] / rate_max,
        df.loc[i, 'month_sin'],
        df.loc[i, 'month_cos'],
        df.loc[i, 'quarter_norm'],
    ]

    img_input = preprocess_input(img.astype(np.float32))
    feat = base_model.predict(np.expand_dims(img_input, 0), verbose=0)
    features_list.append(feat.mean(axis=(1, 2)).flatten())
    meta_list.append(meta)
    targets_list.append(df.loc[i, 'qty'])

    img_tensor = tf.cast(img, tf.float32)[tf.newaxis]
    for _ in range(N_AUG):
        aug = augment(img_tensor, training=True).numpy()[0]
        aug = preprocess_input(aug)
        feat = base_model.predict(np.expand_dims(aug, 0), verbose=0)
        features_list.append(feat.mean(axis=(1, 2)).flatten())
        meta_list.append(meta)
        targets_list.append(df.loc[i, 'qty'])

In [ ]:
X_img  = np.array(features_list)          
X_meta = np.array(meta_list)              
X = np.hstack([X_img, X_meta])       
y = np.log1p(np.array(targets_list, dtype=float))

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (756, 1284), y shape: (756,)


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

n_orig = len(df)
groups = np.repeat(np.arange(n_orig), N_AUG + 1)

model = XGBRegressor(
    n_estimators = 300,
    learning_rate = 0.02,
    max_depth = 4,
    subsample = 0.75,
    colsample_bytree = 0.5,
    reg_lambda = 3.0,
    reg_alpha = 1.0,
    min_child_weight = 5,
    gamma = 0.1,
    random_state = 42,
)


In [ ]:

gkf  = GroupKFold(n_splits=3, shuffle=True)
maes = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), 1):
    model.fit(X[train_idx], y[train_idx])
    preds = np.expm1(model.predict(X[val_idx]))
    actual = np.expm1(y[val_idx])
    mae = mean_absolute_error(actual, preds)   
    maes.append(mae)                             
    print(f"Fold {fold}  MAE: {mae:.1f}")

print(f"\nCV MAE: {np.mean(maes):.1f}")

model.fit(X, y)
print("Final model trained.")

Fold 1  MAE: 8.2
Fold 2  MAE: 36.0
Fold 3  MAE: 15.8

CV MAE: 20.0
Final model trained.


In [ ]:

BLOCKED_CATEGORIES = ["tshirt", "t-shirt", "jeans", "denim", "shirt", "pants", "trousers", "shorts"]

classify_model = tf.keras.applications.MobileNetV2(
    include_top=True, weights='imagenet', input_shape=(224, 224, 3)
)
classify_model.trainable = False

def detect_category(image_path, top_k=5):
    img = cv2.imread(image_path)
    if img is None:
        return []
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = preprocess_input(img.astype(np.float32))
    preds  = classify_model.predict(np.expand_dims(img, 0), verbose=0)
    labels = tf.keras.applications.mobilenet_v2.decode_predictions(preds, top=top_k)
    return [label[1].lower() for label in labels[0]]

# test
print(detect_category(df['image_path'][0]))

['hoopskirt', 'overskirt', 'bonnet', 'gown', 'groom']


In [ ]:
def predict_sales(image_path, rate_value, launch_date):
    img = cv2.imread(image_path)
    if img is None:
        return "Invalid image path"

    top_labels = detect_category(image_path)
    for label in top_labels:
        for blocked in BLOCKED_CATEGORIES:
            if blocked in label:
                return f"Prediction blocked: detected '{label}' (blocked category)"

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(img.astype(np.float32))

    feat = base_model.predict(np.expand_dims(img, 0), verbose=0)
    img_vec = feat.mean(axis=(1, 2)).flatten()

    dt   = pd.to_datetime(launch_date)
    meta = np.array([
        rate_value / rate_max,
        np.sin(2 * np.pi * dt.month / 12),
        np.cos(2 * np.pi * dt.month / 12),
        dt.quarter / 4.0
    ])

    X_input = np.hstack([img_vec, meta]).reshape(1, -1)
    return int(round(max(0, np.expm1(model.predict(X_input)[0]))))

In [ ]:
print("Predicted:", predict_sales(df['image_path'][0], df['rate'][0], df['date'][0]))
print("Actual :", df['qty'][0])

Predicted: 723
Actual   : 1032


In [39]:
import joblib, json, os

os.makedirs("saved_model_2", exist_ok=True)
base_model.save("saved_model_2/mobilenet_extractor.h5")
joblib.dump(model, "saved_model_2/xgb_demand_model.pkl")
with open("saved_model_2/meta.json", "w") as f:
    json.dump({"rate_max": float(rate_max)}, f)

print("Model saved.")

Model saved.
